
#Energy, Momentum, and Collisions using SMT solvers

Most of the work in an introductory mechanics problem is algebra: you know which
conservation laws apply, and then you rearrange them until the unknown is alone on one
side. In this notebook we will look at SMT solvers and see how they let us skip that
step. We will state conservation of energy and conservation of momentum to Z3 exactly
as they are written in a textbook, fix the quantities we know, and let the solver find
the ones we do not. Along the way we will ask it a question that has no answer at all,
and see what it does with that.


**Instructions:**
1. To get started, click on File on the top left and click "Save a copy in Drive."
This will give you an editable version of this document that you can use.
2. If you press `CMD`+`Enter` it runs the cell, and if you press `Shift`+`Enter` it runs the cell and goes to the next one.
3. Make sure you run all cells as you go through the notebook; some cells will not work properly unless the previous one
has been run too.
4. If you disconnect or are inactive for some time you should run all of the cells again.

## 0. Preliminaries (you should run this cell but there is no need to read it)

In [ ]:
!pip install z3-solver
!pip install git+https://github.com/crrivero/FormalMethodsTasting.git#subdirectory=core
from z3 import *
from tofmcore import showSolver, draw_collision
from IPython.display import clear_output
clear_output()

## Encoding constraints in Z3

The goal of this notebook is to teach you about formal methods;
particularly, how you can use existing formal verification tools
(in this case, Z3) to analyze and solve your own problems.
Before we get started, let's look at some basic things we can do with Z3.

### Reals

Let's use Z3 to solve problems involving real numbers. Let's start with something simple: find $x$ such that

$$2x + 5 = 15$$

In [ ]:
# Initialize variables

x = Real('x') # declairing that x is a real number named 'x'

# Initialize Z3 solver
s = Solver()

s.add( 2*x + 5 == 15 ) # add the equation

print(s)
print(s.check())
print(s.model())

Now let's try to check whether that's the only solution. We can do this by adding the following constraint to the solver:

$$x \not= 5$$

If the solver returns "**unsat**" then $x=5$ is the only solution.
Try it yourself by completing the code in the cell below.

In [ ]:
s.add( x == 5 ) # REPLACE THIS LINE
s.check()

## Energy Units

Energy is measured in Joules (kg*m^2/sec^2).

Every quantity we work with in this notebook is a real number, so we will describe
each one to Z3 as a `Real` and then write down the relationships between them as
constraints. We will not rearrange any equations ourselves: we state the physics as
it appears in a textbook, fix the quantities we know, and ask the solver for the
ones we don't.

## Kinetic Energy

Kinetic energy is proportional to an object's mass and velocity.

$$ KE = \frac{1}{2} m v^2 $$

Let's put that one equation into a solver and look at it.

In [ ]:
s = Solver()
KE, m, v = Reals('KE m v')

s.add(KE == (m*v**2)/2)

showSolver(s)

So far the solver knows the relationship but none of the three quantities, so there
are infinitely many assignments that satisfy it. Let's pin two of them down.

What is the kinetic energy of a 5kg mass moving at 2m/sec?

In [ ]:
s.add(m == 5)
s.add(v == 2)

print(s.check())
print(s.model())
print(f"KE = {s.model()[KE]} joules")

"sat" means the constraints can all hold at once, and the model is an assignment
that makes them hold. Here that assignment is the answer we wanted: with the mass and
the velocity fixed, only one value of $KE$ works.

## Potential Energy

Potential energy is proportional to an object's mass and displacement (or height).

$$ PE = m g h $$

The acceleration due to gravity, $g$, is not an unknown, so we give it to Z3 as a
plain Python number rather than as a `Real`.

In [ ]:
s = Solver()
g = 9.81
PE, m, h = Reals('PE m h')

s.add(PE == m*g*h)

showSolver(s)

What is the potential energy of a 5kg mass 20m above the ground?

In [ ]:
s.add(m == 5)
s.add(h == 20)

print(s.check())
print(s.model())
print(f"PE = {s.model()[PE]} joules")

## The Law of Conservation of Energy
Energy is conserved in a closed system.
That is to say, energy can neither be created nor destroyed.
However energy can be transformed from one type to another.
For example, potential energy can become kinetic when a ball is dropped from the air.

Written as a constraint, this says that the total energy at some earlier moment
equals the total energy at some later moment:

$$ KE_1 + PE_1 = KE_2 + PE_2 $$

In [ ]:
s = Solver()
g = 9.81
v1, v2, h1, h2, m1, m2 = Reals('v1 v2 h1 h2 m1 m2')

KE1 = (m1*v1**2)/2
PE1 = m1*g*h1
KE2 = (m2*v2**2)/2
PE2 = m2*g*h2
s.add(KE1 + PE1 == KE2 + PE2)

showSolver(s)

What is the final velocity of a 10kg mass with an initial velocity of 0, an initial height of 20m, and a final height of 10m?

The mass is falling, so its velocity at the later moment points downward. We add that
as a constraint as well, for a reason we will come back to in a moment.

In [ ]:
s.add(m1 == m2)
s.add(m1 == 10)
s.add(v1 == 0)
s.add(h1 == 20)
s.add(h2 == 10)

s.add(v2 < 0) # the mass is on its way down

print(s.check())
print(s.model())
print(f"v2 = {s.model()[v2]} m/s")

The velocity appears in the energy equation as $v^2$, so the equation on its own is
satisfied by two values: about $-14.0$ m/s and about $+14.0$ m/s. Both are real
solutions, and without the constraint we just added, Z3 is free to hand us either
one. Energy accounting fixes the speed; it does not know which way the mass is
travelling. Try deleting the `v2 < 0` line and re-running the cell to see what
happens.

## Momentum

Energy is not the only quantity that survives a closed system. The **momentum** of an
object is its mass times its velocity,

$$ p = m v $$

and unlike energy, momentum carries a direction: a cart moving left has negative
momentum. This turns out to be exactly what we need to work out what happens when two
objects run into each other.

In [ ]:
s = Solver()
p, m, v = Reals('p m v')

s.add(p == m*v)

# a 3 kg cart rolling backwards at 4 m/s
s.add(m == 3)
s.add(v == -4)

print(s.check())
print(f"p = {s.model()[p]} kg*m/s")

## Collisions

Now consider two carts on a track. Cart 1 has a mass of 2kg and is rolling to the
right at 3m/sec. Cart 2 has a mass of 1kg and is sitting still. Cart 1 catches up with
cart 2 and they collide.

Whatever happens during that collision, the total momentum of the two carts together
is the same afterwards as it was before:

$$ m_1 v_{1i} + m_2 v_{2i} = m_1 v_{1f} + m_2 v_{2f} $$

where the $i$ subscripts are the velocities before the collision and the $f$
subscripts are the velocities after.

Let's start with the case where the two carts stick together, which is called a
**perfectly inelastic** collision. Sticking together means they leave with a single
shared velocity, so $v_{1f} = v_{2f}$.

In [ ]:
s = Solver()

m1, m2 = Reals('m_1 m_2')
v1i, v2i = Reals('v_{1i} v_{2i}')
v1f, v2f = Reals('v_{1f} v_{2f}')

# the two carts
s.add(m1 == 2, m2 == 1)
s.add(v1i == 3, v2i == 0)

# momentum is conserved
s.add(m1*v1i + m2*v2i == m1*v1f + m2*v2f)

# the carts stick together, so they leave with the same velocity
s.add(v1f == v2f)

showSolver(s)
print(s.check())

In [ ]:
solution = s.model()
print(solution)
print(f"the carts move off together at {solution[v1f]} m/s")

We have also defined a function to draw the two carts before and after:

In [ ]:
draw_collision(solution[m1], solution[m2],
               solution[v1i], solution[v2i],
               solution[v1f], solution[v2f])

### Is the kinetic energy conserved too?

We now have the velocity after the collision, so we can ask a different question. Is
the kinetic energy of the two carts the same afterwards as it was before?

$$ \frac{1}{2} m_1 v_{1i}^2 + \frac{1}{2} m_2 v_{2i}^2 = \frac{1}{2} m_1 v_{1f}^2 + \frac{1}{2} m_2 v_{2f}^2 $$

Rather than computing both sides and comparing them, we can hand the claim to the
solver as one more constraint and see whether it can still satisfy everything at
once.

In [ ]:
s.add( (m1*v1i**2)/2 + (m2*v2i**2)/2 == (m1*v1f**2)/2 + (m2*v2f**2)/2 )

print(s.check())

"unsat" means there is no assignment satisfying all of our constraints together.
The carts, their masses, momentum conservation and sticking together were all
consistent a moment ago, so the constraint we just added is the one that cannot hold:
**kinetic energy is not conserved in this collision**. Some of it went into deforming
the carts and into heat and sound.

This is worth pausing on. We did not derive that result, and we did not check it by
arithmetic. We stated the physics we were confident about, asserted the claim we were
unsure about, and the solver told us the two are incompatible.

### Elastic collisions

Now suppose the carts bounce off each other cleanly instead of sticking, with nothing
lost to heat or deformation. A collision like that is called **elastic**, and in an
elastic collision the kinetic energy *is* conserved.

The setup below is the same two carts, and momentum conservation has already been
written for you. **Replace the marked line** with the constraint saying that the total
kinetic energy of the two carts after the collision equals the total kinetic energy
before it.

Note the last constraint: leaving the carts untouched, with $v_{1f} = 3$ and
$v_{2f} = 0$, conserves both momentum and kinetic energy perfectly well, so we have to
tell the solver that cart 1 does not simply carry on as if nothing happened.

In [ ]:
s = Solver()

m1, m2 = Reals('m_1 m_2')
v1i, v2i = Reals('v_{1i} v_{2i}')
v1f, v2f = Reals('v_{1f} v_{2f}')

# the two carts, exactly as before
s.add(m1 == 2, m2 == 1)
s.add(v1i == 3, v2i == 0)

# momentum is conserved
s.add(m1*v1i + m2*v2i == m1*v1f + m2*v2f)

# kinetic energy is conserved
s.add( True ) # REPLACE THIS LINE

# rule out the "the carts never touched" solution
s.add(v1f != v1i)

showSolver(s)
print(s.check())

In [ ]:
solution = s.model()
print(solution)
print(f"cart 1 leaves at {solution[v1f]} m/s and cart 2 leaves at {solution[v2f]} m/s")

In [ ]:
draw_collision(solution[m1], solution[m2],
               solution[v1i], solution[v2i],
               solution[v1f], solution[v2f])

Cart 1 keeps moving to the right, but more slowly, and cart 2 is pushed out ahead of
it faster than cart 1 was ever going. Both carts together still carry the same
momentum and the same kinetic energy as cart 1 carried on its own at the start.


###Congratulations! You just used an SMT solver to work out what happens when two carts collide!


####If you'd like to continue your Z3 journey, you can start with this guide to learn more:
https://ericpony.github.io/z3py-tutorial/guide-examples.htm